# CIC-IDS2017: контрольні експерименти на виправленому (AM/PM) спліті

Відтворює заморожений протокол Section 3.7/5.2 (ті самі LightGBM-гіперпараметри,
Table 5, той самий quantile(0.70/0.85) спліт), але на ВИПРАВЛЕНИХ таймстемпах,
і додає три контролі, потрібні щоб зрозуміти, чи ΔAP на CIC — це CTI, чи просто порт/новий клас атаки:

1. **F0 + dst_port** — чи дає сам номер порту (без жодного CTI) той самий ефект;
2. **Placebo-CTI** — F1 зі статистиками CVE, перемішаними між рядками (та сама форма ознак, нуль смислового зв'язку);
3. **Recall окремо по DDoS і PortScan** — чи весь приріст сконцентрований саме в класах атак, відсутніх у train.

Нічого не змінює у вихідних файлах — лише читає і друкує/зберігає CSV з результатами.

In [1]:
# 0. Автопошук CTI-збагаченого файлу CIC-IDS2017 (cve_ctx_* колонки) — не вгадуємо назву,
# а шукаємо під /srv/ids_research/data/enriched/, за аналогією з UNSW/GeNIS enriched-файлами
import subprocess
res = subprocess.run(
    ["find", "/srv/ids_research/data/enriched", "/srv/ids_research/data",
     "/home/ivan_r_opirskyy", "-iname", "*cic*", "-type", "f"],
    capture_output=True, text=True, timeout=60
)
print(res.stdout)
print(res.stderr)
# Знайдіть серед результатів файл з 'cve_context' / 'enriched' у назві (саме з нього F1
# будувався у статті) і підставте його шлях у клітинку нижче замість PATH.

/srv/ids_research/data/enriched/cic_ids2017_with_cve_context.parquet
/srv/ids_research/data/_source_check/CIC-IDS2017/Network-Flows/CICIDS_Flow.parquet
/srv/ids_research/data/_source_check/CIC-IDS2017/.cache/huggingface/download/Network-Flows/CICIDS_Flow.parquet.metadata
/srv/ids_research/data/_source_check/CIC-IDS2017/.cache/huggingface/download/Network-Flows/CICIDS_Flow.parquet.lock
/srv/ids_research/data/enriched/cic_ids2017_with_cve_context.parquet
/srv/ids_research/data/cic_control_experiments_summary.csv
/srv/ids_research/data/cic_ids2017_flows.parquet
/srv/ids_research/data/cic_ids2017_flows_fixed.parquet
/srv/ids_research/data/preprocessed/scaled/CIC-IDS-2017.parquet
/srv/ids_research/data/preprocessed/scaled/CICIoT2023.parquet
/srv/ids_research/data/preprocessed/scaled/CIC-YNU-IoTMal-2026.parquet
/srv/ids_research/data/preprocessed/meta/CIC-YNU-IoTMal-2026_unknown.parquet
/srv/ids_research/data/preprocessed/meta/CICIoT2023_scaler.pkl
/srv/ids_research/data/preprocessed/meta/CI

In [2]:
import pandas as pd, numpy as np, lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, f1_score

PATH = "/srv/ids_research/data/enriched/cic_ids2017_with_cve_context.parquet"
TS_COL = "Timestamp"
LABEL_COL = "attack_label"
PORT_COL_CANDIDATES = ["Destination Port", "destination_port", "dst_port", "Dst Port", "dstport"]
SEED = 42

# Заморожений контракт ознак (Section 3, F0_CIC = 77 предикторів; EXACT ORDER, project-memory confirmed)
F0_CIC = ['Flow Duration','Total Fwd Packets','Total Backward Packets','Total Length of Fwd Packets',
'Total Length of Bwd Packets','Fwd Packet Length Max','Fwd Packet Length Min','Fwd Packet Length Mean',
'Fwd Packet Length Std','Bwd Packet Length Max','Bwd Packet Length Min','Bwd Packet Length Mean',
'Bwd Packet Length Std','Flow Bytes/s','Flow Packets/s','Flow IAT Mean','Flow IAT Std','Flow IAT Max',
'Flow IAT Min','Fwd IAT Total','Fwd IAT Mean','Fwd IAT Std','Fwd IAT Max','Fwd IAT Min','Bwd IAT Total',
'Bwd IAT Mean','Bwd IAT Std','Bwd IAT Max','Bwd IAT Min','Fwd PSH Flags','Bwd PSH Flags','Fwd URG Flags',
'Bwd URG Flags','Fwd Header Length','Bwd Header Length','Fwd Packets/s','Bwd Packets/s','Min Packet Length',
'Max Packet Length','Packet Length Mean','Packet Length Std','Packet Length Variance','FIN Flag Count',
'SYN Flag Count','RST Flag Count','PSH Flag Count','ACK Flag Count','URG Flag Count','CWE Flag Count',
'ECE Flag Count','Down/Up Ratio','Average Packet Size','Avg Fwd Segment Size','Avg Bwd Segment Size',
'Fwd Avg Bytes/Bulk','Fwd Avg Packets/Bulk','Fwd Avg Bulk Rate','Bwd Avg Bytes/Bulk','Bwd Avg Packets/Bulk',
'Bwd Avg Bulk Rate','Subflow Fwd Packets','Subflow Fwd Bytes','Subflow Bwd Packets','Subflow Bwd Bytes',
'Init_Win_bytes_forward','Init_Win_bytes_backward','act_data_pkt_fwd','min_seg_size_forward','Active Mean',
'Active Std','Active Max','Active Min','Idle Mean','Idle Std','Idle Max','Idle Min','protocol']
CVE_CTX = ['cve_ctx_service','cve_ctx_pool_size','cve_ctx_n_epss','cve_ctx_epss_coverage',
           'cve_ctx_service_max_epss','cve_ctx_n_generic_os_matches','cve_ctx_n_specific_product_matches']
F1_CIC = F0_CIC + CVE_CTX
CAT_COLS = {"protocol", "cve_ctx_service"}

df = pd.read_parquet(PATH)
df.columns = [c.strip() for c in df.columns]
PORT_COL = next((c for c in PORT_COL_CANDIDATES if c in df.columns), None)
if PORT_COL is None:
    port_like = [c for c in df.columns if "port" in c.lower()]
    print("PORT_COL не знайдено серед кандидатів. Колонки з 'port' у назві:", port_like)
    print("=> підставте правильну назву в PORT_COL_CANDIDATES і перезапустіть цю клітинку.")
else:
    print("PORT_COL визначено як:", PORT_COL)

print("rows:", len(df), "| cols:", len(df.columns))
expected_cols = set(F1_CIC) | ({PORT_COL} if PORT_COL else set())
missing = [c for c in expected_cols if c not in df.columns]
print("відсутні очікувані колонки:", missing if missing else "— жодної, все на місці")


PORT_COL визначено як: destination_port
rows: 2827677 | cols: 96
відсутні очікувані колонки: — жодної, все на місці


In [3]:
# 1b. Перевірка: звіряємо хардкоджені F0_CIC/F1_CIC з реальними замороженими контрактами на сервері
import json, os
contract_dir = "/home/ivan_r_opirskyy/notebooks/2 article/release/results/step5d2_audit"
for name, hardcoded in [("feature_contract_CIC_F0.json", F0_CIC), ("feature_contract_CIC_F1.json", F1_CIC)]:
    path = os.path.join(contract_dir, name)
    if not os.path.exists(path):
        print(f"[!] {name} не знайдено за {path} — звірку пропущено")
        continue
    real = json.load(open(path))
    # Структура файлу невідома наперед — знаходимо список ознак, не вгадуючи назву ключа
    if isinstance(real, list):
        real_list = real
    elif isinstance(real, dict):
        list_candidates = {k: v for k, v in real.items() if isinstance(v, list)}
        if len(list_candidates) == 1:
            real_list = next(iter(list_candidates.values()))
        else:
            print(f"[?] {name}: не вдалось однозначно визначити список ознак. Ключі верхнього рівня: {list(real.keys())}")
            print("    Повний вміст (перші 2000 симв.):", json.dumps(real, ensure_ascii=False)[:2000])
            continue
    else:
        print(f"[?] {name}: неочікуваний тип {type(real)}")
        continue
    if list(real_list) == list(hardcoded):
        print(f"[OK] {name}: збігається точно, {len(real_list)} ознак")
    else:
        print(f"[MISMATCH] {name}: сервер={len(real_list)} ознак, хардкод={len(hardcoded)} ознак")
        print("  лише на сервері:", [c for c in real_list if c not in hardcoded])
        print("  лише в хардкоді:", [c for c in hardcoded if c not in real_list])
        print("  !! ВИПРАВТЕ F0_CIC/F1_CIC вище на реальний список перед продовженням !!")

[OK] feature_contract_CIC_F0.json: збігається точно, 77 ознак
[OK] feature_contract_CIC_F1.json: збігається точно, 84 ознак


In [4]:
# 1. Виправлення AM/PM (+12 год для годин 0-7, підтверджено по офіційному розкладу CIC)
raw = df[TS_COL]
ts = raw if np.issubdtype(raw.dtype, np.datetime64) else pd.to_datetime(raw.astype(str), dayfirst=True, errors="coerce")
fixed_ts = ts.where(ts.dt.hour >= 8, ts + pd.Timedelta(hours=12))
df["_ts"] = fixed_ts
print("виправлений діапазон:", fixed_ts.min(), "—", fixed_ts.max())

виправлений діапазон: 2017-07-03 08:55:58 — 2017-07-07 17:02:00


In [5]:
# 2. Той самий спліт-протокол, що й у статті (quantile 0.70/0.85, snap до найближчого
# реально наявного значення), але на виправлених таймстемпах
def quantile_split(t, q1=0.70, q2=0.85):
    b1 = t.quantile(q1, interpolation="nearest")
    b2 = t.quantile(q2, interpolation="nearest")
    part = np.where(t < b1, "train", np.where(t < b2, "val", "test"))
    return pd.Series(part, index=t.index), b1, b2

df["_split"], b1, b2 = quantile_split(df["_ts"])
print("нові межі (виправлені):", b1, "|", b2)
ct = pd.crosstab(df[LABEL_COL], df["_split"])[["train", "val", "test"]]
print(ct.to_string())
print("\nATTACK-share:", {s: round(100 * (df.loc[df['_split']==s, LABEL_COL] != 'BENIGN').mean(), 2) for s in ['train','val','test']})

нові межі (виправлені): 2017-07-06 15:14:00 | 2017-07-07 14:51:00
_split                        train     val    test
attack_label                                       
BENIGN                      1707824  425493  137805
Bot                               0    1956       0
DDoS                              0       0  128025
DoS GoldenEye                 10293       0       0
DoS Hulk                     230123       0       0
DoS Slowhttptest               5499       0       0
DoS slowloris                  5796       0       0
FTP-Patator                    7935       0       0
Heartbleed                       11       0       0
Infiltration                     22      14       0
PortScan                          0     372  158432
SSH-Patator                    5897       0       0
Web Attack – Brute Force       1507       0       0
Web Attack – Sql Injection       21       0       0
Web Attack – XSS                652       0       0

ATTACK-share: {'train': 13.55, 'val': 0.55, 'test

In [6]:
# 3. Допоміжні функції: категорії fit лише на train, бінарна мета, кастомна AP-feval,
# заморожені гіперпараметри (Table 5)
y = (df[LABEL_COL] != "BENIGN").astype(int)

def fit_categories(frame, cols, train_mask):
    out = frame.copy()
    for c in cols:
        if c not in out.columns:
            continue
        cats = pd.Categorical(out.loc[train_mask, c]).categories
        out[c] = pd.Categorical(out[c], categories=cats)
    return out

def ap_feval(preds, data):
    labels = data.get_label()
    return "unweighted_ap", average_precision_score(labels, preds), True

LGB_PARAMS = dict(objective="binary", metric="None", seed=SEED, num_leaves=15,
                  learning_rate=0.05, min_data_in_leaf=50, verbosity=-1)

def train_eval(feature_cols, cat_cols, tag, label_override=None):
    cat_cols = [c for c in cat_cols if c in feature_cols]
    frame = fit_categories(df[feature_cols], cat_cols, df["_split"] == "train")
    yy = label_override if label_override is not None else y
    tr = df["_split"] == "train"; va = df["_split"] == "val"; te = df["_split"] == "test"
    n_pos, n_neg = yy[tr].sum(), (yy[tr] == 0).sum()
    params = dict(LGB_PARAMS, scale_pos_weight=n_neg / n_pos)
    dtr = lgb.Dataset(frame[tr], label=yy[tr], categorical_feature=cat_cols, free_raw_data=False)
    dva = lgb.Dataset(frame[va], label=yy[va], categorical_feature=cat_cols, reference=dtr, free_raw_data=False)
    model = lgb.train(params, dtr, num_boost_round=500, valid_sets=[dva], feval=ap_feval,
                       callbacks=[lgb.early_stopping(30, verbose=False)])
    pred_te = model.predict(frame[te], num_iteration=model.best_iteration)
    ap = average_precision_score(yy[te], pred_te)
    auc = roc_auc_score(yy[te], pred_te)
    pred_bin = (pred_te >= 0.5).astype(int)
    print(f"[{tag}] n_features={len(feature_cols)} best_iter={model.best_iteration} "
          f"test_AP={ap:.6f} test_AUC={auc:.6f} "
          f"P={precision_score(yy[te], pred_bin, zero_division=0):.4f} "
          f"R={recall_score(yy[te], pred_bin, zero_division=0):.4f}")
    return model, pred_te, te, ap, auc

In [7]:
# 4. Базові моделі на виправленому спліті: F0 і F1 (справжній CTI)
m_f0, p_f0, mask_te, ap_f0, auc_f0 = train_eval(F0_CIC, CAT_COLS, "F0 (fixed-ts)")
m_f1, p_f1, _,        ap_f1, auc_f1 = train_eval(F1_CIC, CAT_COLS, "F1 (fixed-ts)")
print(f"\nΔAP (F1-F0) на виправленому спліті: {ap_f1 - ap_f0:+.6f}  (у статті, на НЕВИПРАВЛЕНОМУ спліті: +0.263659)")

[F0 (fixed-ts)] n_features=77 best_iter=11 test_AP=0.896485 test_AUC=0.838062 P=0.1652 R=0.0002
[F1 (fixed-ts)] n_features=84 best_iter=163 test_AP=0.937376 test_AUC=0.884470 P=0.9993 R=0.2827

ΔAP (F1-F0) на виправленому спліті: +0.040891  (у статті, на НЕВИПРАВЛЕНОМУ спліті: +0.263659)


In [8]:
# 5. Контроль 1 — F0 + dst_port (виключений з F0 за протоколом; тут додається як сирий номер порту)
F0_PORT = F0_CIC + [PORT_COL]
m_port, p_port, _, ap_port, auc_port = train_eval(F0_PORT, CAT_COLS, "F0+dst_port (fixed-ts)")
print(f"\nΔAP (F0+port - F0): {ap_port - ap_f0:+.6f}  |  частка від повного ΔAP(F1-F0): "
      f"{100*(ap_port - ap_f0)/(ap_f1 - ap_f0):.1f}%  (якщо >> 50% — ефект пояснює переважно порт, не CTI)")

[F0+dst_port (fixed-ts)] n_features=78 best_iter=97 test_AP=0.887833 test_AUC=0.773922 P=0.9980 R=0.2827

ΔAP (F0+port - F0): -0.008652  |  частка від повного ΔAP(F1-F0): -21.2%  (якщо >> 50% — ефект пояснює переважно порт, не CTI)


In [9]:
# 6. Контроль 2 — Placebo-CTI: той самий F1, але значення cve_ctx_* перемішані (shuffle)
# ОКРЕМО в межах кожної партиції (train/val/test), щоб зберегти маргінальний розподіл і
# розмір датасету, але зруйнувати будь-який справжній зв'язок рядок↔контекст
rng = np.random.default_rng(SEED)
df_placebo = df.copy()
for part in ["train", "val", "test"]:
    idx = df_placebo.index[df_placebo["_split"] == part]
    perm = rng.permutation(idx)
    df_placebo.loc[idx, CVE_CTX] = df.loc[perm, CVE_CTX].values

_orig_df = df
df = df_placebo   # train_eval/fit_categories читають глобальний df — підміняємо на час цього блоку
m_plac, p_plac, _, ap_plac, auc_plac = train_eval(F1_CIC, CAT_COLS, "F1-placebo (shuffled CVE ctx)")
df = _orig_df     # повертаємо справжній df

print(f"\nΔAP (placebo - F0): {ap_plac - ap_f0:+.6f}  |  ΔAP (справжній F1 - F0): {ap_f1 - ap_f0:+.6f}")
print("Якщо ці два приросту близькі — ефект дає сама ФОРМА/наявність 7 додаткових колонок")
print("(тобто додаткова ємність моделі чи indicator of 'this flow has cve_ctx data'),")
print("а не змістовний зв'язок CVE/EPSS зі службою. Якщо placebo << справжній F1 — контент CTI має значення.")

[F1-placebo (shuffled CVE ctx)] n_features=84 best_iter=11 test_AP=0.896485 test_AUC=0.838062 P=0.1652 R=0.0002

ΔAP (placebo - F0): +0.000000  |  ΔAP (справжній F1 - F0): +0.040891
Якщо ці два приросту близькі — ефект дає сама ФОРМА/наявність 7 додаткових колонок
(тобто додаткова ємність моделі чи indicator of 'this flow has cve_ctx data'),
а не змістовний зв'язок CVE/EPSS зі службою. Якщо placebo << справжній F1 — контент CTI має значення.


In [10]:
# 7. Контроль 3 — recall окремо по DDoS і PortScan (і решті класів) для F0 vs F1 vs F0+port vs placebo
def per_class_recall(pred, mask, tag, thr=0.5):
    sub_labels = df.loc[mask, LABEL_COL]
    pred_bin = (pred >= thr).astype(int)
    rows = []
    for cls in sub_labels.unique():
        if cls == "BENIGN":
            continue
        m = (sub_labels == cls).to_numpy(dtype=bool)
        if m.sum() == 0:
            continue
        rows.append((cls, int(m.sum()), recall_score(np.ones(m.sum()), pred_bin[m])))
    out = pd.DataFrame(rows, columns=["attack_label", "n_test", f"recall_{tag}"]).set_index("attack_label")
    return out

r_f0   = per_class_recall(p_f0,   mask_te, "F0")
r_f1   = per_class_recall(p_f1,   mask_te, "F1")
r_port = per_class_recall(p_port, mask_te, "F0_port")
r_plac = per_class_recall(p_plac, mask_te, "placebo")

table = r_f0.join(r_f1.drop(columns="n_test")).join(r_port.drop(columns="n_test")).join(r_plac.drop(columns="n_test"))
print(table.round(4).to_string())
print("\nОсобливо подивіться на рядки DDoS і PortScan — якщо recall_F1 >> recall_F0 САМЕ там,")
print("а по решті класів F0≈F1, це підтверджує, що ΔAP concentration на класах, відсутніх у train.")

              n_test  recall_F0  recall_F1  recall_F0_port  recall_placebo
attack_label                                                              
PortScan      158432     0.0002     0.0024          0.0023          0.0002
DDoS          128025     0.0001     0.6296          0.6298          0.0001

Особливо подивіться на рядки DDoS і PortScan — якщо recall_F1 >> recall_F0 САМЕ там,
а по решті класів F0≈F1, це підтверджує, що ΔAP concentration на класах, відсутніх у train.


In [11]:
# 8. Підсумкова таблиця і збереження результатів
summary = pd.DataFrame([
    {"model": "F0",          "n_features": len(F0_CIC),        "test_AP": ap_f0,   "test_AUC": auc_f0},
    {"model": "F0+dst_port", "n_features": len(F0_PORT),        "test_AP": ap_port, "test_AUC": auc_port},
    {"model": "F1 (real CTI)","n_features": len(F1_CIC),        "test_AP": ap_f1,   "test_AUC": auc_f1},
    {"model": "F1-placebo",  "n_features": len(F1_CIC),        "test_AP": ap_plac, "test_AUC": auc_plac},
])
summary["delta_AP_vs_F0"] = summary["test_AP"] - ap_f0
print(summary.round(6).to_string(index=False))

summary.to_csv("/srv/ids_research/data/cic_control_experiments_summary.csv", index=False)
table.to_csv("/srv/ids_research/data/cic_control_per_class_recall.csv")
print("\nЗбережено: cic_control_experiments_summary.csv, cic_control_per_class_recall.csv")

        model  n_features  test_AP  test_AUC  delta_AP_vs_F0
           F0          77 0.896485  0.838062        0.000000
  F0+dst_port          78 0.887833  0.773922       -0.008652
F1 (real CTI)          84 0.937376  0.884470        0.040891
   F1-placebo          84 0.896485  0.838062        0.000000

Збережено: cic_control_experiments_summary.csv, cic_control_per_class_recall.csv


## Додаткові контролі: що саме дає приріст F1 — групування за сервісом чи зміст CVE/EPSS?
Placebo вище перемішував УСІ 7 колонок, включно з `cve_ctx_service` (яка похідна від порту),
тому він не розділяє «групування трафіку за сервісом» і «значення CVE/EPSS». Ці клітинки розділяють.
Плюс перевірка, чи не є ΔAP артефактом early stopping на val з 0.55% атак.

In [12]:
# 9. Базова лінія AP і чи є числові CTI-ознаки константами всередині сервісу
te_mask = df["_split"] == "test"
print("AP випадкового класифікатора на test (= частка атак):", round(float(y[te_mask].mean()), 6))
NUM_CTX = [c for c in CVE_CTX if c != "cve_ctx_service"]
nuniq = df.groupby("cve_ctx_service", observed=True)[NUM_CTX].nunique(dropna=False).max()
print("\nМакс. к-сть різних значень кожної числової CTI-ознаки в межах одного сервісу")
print("(1 = ознака повністю визначається сервісом):")
print(nuniq.to_string())

AP випадкового класифікатора на test (= частка атак): 0.675189

Макс. к-сть різних значень кожної числової CTI-ознаки в межах одного сервісу
(1 = ознака повністю визначається сервісом):
cve_ctx_pool_size                     1
cve_ctx_n_epss                        1
cve_ctx_epss_coverage                 1
cve_ctx_service_max_epss              1
cve_ctx_n_generic_os_matches          1
cve_ctx_n_specific_product_matches    1


In [13]:
# 10. Розкладання F1: лише сервіс vs лише числові CVE/EPSS-ознаки
m_svc, p_svc, _, ap_svc, auc_svc = train_eval(F0_CIC + ["cve_ctx_service"], CAT_COLS, "F0 + cve_ctx_service only")
m_num, p_num, _, ap_num, auc_num = train_eval(F0_CIC + NUM_CTX, CAT_COLS, "F0 + numeric CVE/EPSS only")

[F0 + cve_ctx_service only] n_features=78 best_iter=75 test_AP=0.820815 test_AUC=0.700308 P=0.9979 R=0.2799
[F0 + numeric CVE/EPSS only] n_features=83 best_iter=75 test_AP=0.899555 test_AUC=0.765650 P=0.9978 R=0.2863


In [14]:
# 11. Service-level placebo: групування за сервісом ЗБЕРЕЖЕНЕ, але кожному сервісу
# присвоєно CVE/EPSS-статистики ІНШОГО сервісу. Якщо AP не падає — важить групування,
# а не самі значення EPSS/CVE. Якщо падає — зміст CTI має значення.
key = df["cve_ctx_service"].astype(str)
svc_stats = df.groupby(key, observed=True)[NUM_CTX].first()
svc_stats = svc_stats[~svc_stats.index.isin(["nan", "<NA>", "None"])]
rng2 = np.random.default_rng(SEED)
perm_idx = rng2.permutation(len(svc_stats))
df_svcperm = df.copy()
for c in NUM_CTX:
    mapping = dict(zip(svc_stats.index, svc_stats[c].to_numpy()[perm_idx]))
    mapped = key.map(mapping)
    df_svcperm[c] = pd.to_numeric(mapped.where(key.isin(list(mapping.keys())), df[c]), errors="coerce")
print("сервісів переставлено:", len(svc_stats))

_orig_df = df
df = df_svcperm
m_sp, p_sp, _, ap_sp, auc_sp = train_eval(F0_CIC + NUM_CTX, CAT_COLS, "F0 + numeric CVE/EPSS, permuted across services")
df = _orig_df

print(f"\nAP: numeric real = {ap_num:.6f} | numeric permuted across services = {ap_sp:.6f} | різниця = {ap_num - ap_sp:+.6f}")

сервісів переставлено: 29
[F0 + numeric CVE/EPSS, permuted across services] n_features=83 best_iter=2 test_AP=0.810281 test_AUC=0.693823 P=0.0000 R=0.0000

AP: numeric real = 0.899555 | numeric permuted across services = 0.810281 | різниця = +0.089274


In [15]:
# 12. Чи не є ΔAP артефактом early stopping (val має лише 0.55% атак)?
# Тренуємо F0 і F1 з ФІКСОВАНОЮ кількістю дерев, без early stopping.
def train_fixed(feature_cols, rounds):
    cat_cols = [c for c in CAT_COLS if c in feature_cols]
    frame = fit_categories(df[feature_cols], cat_cols, df["_split"] == "train")
    tr = df["_split"] == "train"; te = df["_split"] == "test"
    n_pos, n_neg = y[tr].sum(), (y[tr] == 0).sum()
    params = dict(LGB_PARAMS, scale_pos_weight=n_neg / n_pos)
    model = lgb.train(params, lgb.Dataset(frame[tr], label=y[tr], categorical_feature=cat_cols),
                      num_boost_round=rounds)
    p = model.predict(frame[te])
    return average_precision_score(y[te], p), roc_auc_score(y[te], p)

rows = []
for rounds in [11, 50, 163, 300, 500]:
    a0, u0 = train_fixed(F0_CIC, rounds)
    a1, u1 = train_fixed(F1_CIC, rounds)
    rows.append({"rounds": rounds, "AP_F0": a0, "AP_F1": a1, "dAP": a1 - a0, "AUC_F0": u0, "AUC_F1": u1})
    print(f"rounds={rounds:>3}  AP_F0={a0:.6f}  AP_F1={a1:.6f}  dAP={a1 - a0:+.6f}")
fixed_df = pd.DataFrame(rows)

rounds= 11  AP_F0=0.896485  AP_F1=0.805184  dAP=-0.091301
rounds= 50  AP_F0=0.883409  AP_F1=0.819430  dAP=-0.063979
rounds=163  AP_F0=0.932613  AP_F1=0.937376  dAP=+0.004764
rounds=300  AP_F0=0.926477  AP_F1=0.914647  dAP=-0.011830
rounds=500  AP_F0=0.901064  AP_F1=0.893746  dAP=-0.007319


In [16]:
# 13. Підсумок додаткових контролів
extra = pd.DataFrame([
    {"model": "F0",                                "test_AP": ap_f0,  "test_AUC": auc_f0},
    {"model": "F0 + cve_ctx_service only",         "test_AP": ap_svc, "test_AUC": auc_svc},
    {"model": "F0 + numeric CVE/EPSS only",        "test_AP": ap_num, "test_AUC": auc_num},
    {"model": "F0 + numeric, permuted across svc", "test_AP": ap_sp,  "test_AUC": auc_sp},
    {"model": "F1 (full real CTI)",                "test_AP": ap_f1,  "test_AUC": auc_f1},
])
extra["delta_AP_vs_F0"] = extra["test_AP"] - ap_f0
print(extra.round(6).to_string(index=False))
print()
print(fixed_df.round(6).to_string(index=False))
extra.to_csv("/srv/ids_research/data/cic_control_extra_decomposition.csv", index=False)
fixed_df.to_csv("/srv/ids_research/data/cic_control_fixed_rounds.csv", index=False)
print("\nЗбережено: cic_control_extra_decomposition.csv, cic_control_fixed_rounds.csv")

                            model  test_AP  test_AUC  delta_AP_vs_F0
                               F0 0.896485  0.838062        0.000000
        F0 + cve_ctx_service only 0.820815  0.700308       -0.075670
       F0 + numeric CVE/EPSS only 0.899555  0.765650        0.003069
F0 + numeric, permuted across svc 0.810281  0.693823       -0.086205
               F1 (full real CTI) 0.937376  0.884470        0.040891

 rounds    AP_F0    AP_F1       dAP   AUC_F0   AUC_F1
     11 0.896485 0.805184 -0.091301 0.838062 0.684710
     50 0.883409 0.819430 -0.063979 0.773192 0.701100
    163 0.932613 0.937376  0.004764 0.880432 0.884470
    300 0.926477 0.914647 -0.011830 0.867350 0.834120
    500 0.901064 0.893746 -0.007319 0.822987 0.792514

Збережено: cic_control_extra_decomposition.csv, cic_control_fixed_rounds.csv
